<a href="https://colab.research.google.com/github/Ayush-Singh-36/taekwondo_power_prediction_pytorch/blob/secondary/power_calculating_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Development

# Device-Agnostic Code

In [1]:
import torch
from torch import cuda
device = "cuda" if cuda.is_available() else "cpu"
print(device)

cuda


# Importing dataset from kaggle

In [2]:
import os
from google.colab import userdata
import sys
def custom_exit(status):
    print(f"Kaggle API tried to exit with status {status}. Ignoring for Colab environment.")
sys.exit = custom_exit
exit = custom_exit
try:
    __builtins__.exit = custom_exit
except AttributeError:
    print("Could not patch __builtins__.exit - it might not be present or modifiable in this environment.")

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "weyoungopenlab/taekwondo-athlete-anaerobic-power"
download_path = "./data"

print("Authenticating via environment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading movie dataset from Kaggle...")
api.dataset_download_files(dataset_slug, path=download_path, unzip=True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")


Authenticating via environment variables...
Dataset URL: https://www.kaggle.com/datasets/weyoungopenlab/taekwondo-athlete-anaerobic-power
Done! Your files have been saved to the './data' folder.


# Checking for cardinality of the data

In [3]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    # 1. Separate column types automatically
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    # 2. Extract Options from Categorical Columns
    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        # High cardinality warning (e.g., IDs, Hash keys, open text fields)
        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            # Print value distributions so you know if an option is incredibly rare
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    # 3. Extract Ranges from Numerical Columns
    # Using describe gives you min, max, and percentiles to spot extreme values or outliers
    numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
    display(numeric_summary)

# --- Example of running it on your data ---
data = pd.read_csv("/content/data/data.csv")
profile_dataset_features(data)

=== Dataset Shape: 320 rows | 22 columns ===

Found 1 Categorical columns and 21 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'subject_id' | Unique Values Count: 320 | Missing: 0 rows
  ⚠️ High Cardinality! Showing first 5 options sample: ['TKD001', 'TKD002', 'TKD003', 'TKD004', 'TKD005']...

--------------------------------------------------
NUMERICAL FEATURE BOUNDARY DISCOVERY
--------------------------------------------------


,min,max,mean
gender,0.00,1.00,0.553125
age_years,16.00,30.00,21.490625
training_years,3.00,16.00,8.581250
height_cm,155.00,191.96,171.218375
weight_kg,45.00,95.79,65.420687
bmi,16.98,27.40,22.126219
body_fat_pct,5.00,25.00,13.614844
lean_mass_kg,35.91,87.97,56.858062
leg_length_cm,77.52,101.84,89.063187
thigh_circumference_cm,47.81,63.79,55.474469


# Splitting data into training, validation and test set

In [7]:
from sklearn.model_selection import train_test_split

x = data.drop(['subject_id', 'high_anaerobic_power'], axis=1)
y = data['high_anaerobic_power']

x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, shuffle=True, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, shuffle=True, random_state=42)

# Scaling the data

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_x_train = scaler.fit_transform(x_train)
scaled_x_val = scaler.transform(x_val)
scaled_x_test = scaler.transform(x_test)

# Creating dataloaders for further procedure

In [10]:
import torch
from torch.utils.data import TensorDataset, DataLoader

x_train_tensor = torch.tensor(scaled_x_train, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).unsqueeze(1)
x_val_tensor = torch.tensor(scaled_x_val, dtype = torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype = torch.float32).unsqueeze(1)
x_test_tensor = torch.tensor(scaled_x_test, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).unsqueeze(1)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

BATCH_SIZE = 16

train_loader = DataLoader(dataset = train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(dataset = val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(dataset = test_dataset, batch_size = BATCH_SIZE, shuffle = False)